# Notebook 4: Taxonomic Assignment
In this notebook, we will assign taxonomic classifications to our 6,013 ASVs using the reference database (SILVA). This step maps our biological sequences to known bacterial taxonomies from Kingdom down to Genus level.

In [ ]:
import os
import urllib.request

project_dir = "/home/azureuser/Microbiome_project"
output_path = os.path.join(project_dir, "silva_nr99_v138.1_train_set.fa.gz")

url = "https://zenodo.org/record/4587955/files/silva_nr99_v138.1_train_set.fa.gz"

print("Starting download of SILVA database (v138.1)... Please wait.")
try:
  urllib.request.urlretrieve(url, output_path)
  print("Download completed successfully!")
  print("File saved at:", output_path)
  print("File size:", os.path.getsize(output_path), "bytes")
except Exception as e:
  print("Download failed:", e)

In [ ]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
library(dada2)

# 1. Load the final ASV table
seqtab.nochim <- readRDS("seqtab_final.rds")

# 2. Specify the path to the downloaded SILVA training fasta file
tax_db <- "silva_nr99_v138.1_train_set.fa.gz"

cat("Assigning taxonomy using SILVA database...\n")
taxa <- assignTaxonomy(seqtab.nochim, tax_db, multithread = TRUE, verbose = TRUE)

# 3. Save the taxonomy object
saveRDS(taxa, "taxa_final.rds")
cat("Taxonomy assignment completed and saved to taxa_final.rds!\n")

# Print a preview of the taxonomic assignments
taxa.print <- taxa
rownames(taxa.print) <- NULL
head(taxa.print, 10)
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)